In [2]:
# Findings summary
# Reads: ../data/clean_joined.csv 

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm


In [3]:
# I load the joined table
df = pd.read_csv(r"C:\Users\admin\Desktop\Captoneproject-pydata\data\clean_joined.csv")

In [4]:
# recompute every number I want to quote, so my summary always matches my data
r = {}

In [5]:
# the data
r["rows"] = len(df)
r["countries"] = df["Entity"].nunique()
r["first_year"], r["last_year"] = df["Year"].min(), df["Year"].max()

In [6]:
# correlations
r["r_life"] = stats.pearsonr(df["log_gdp"], df["life_exp"])[0]
r["r_mort"] = stats.pearsonr(df["log_gdp"], df["child_mort"])[0]
r["r_mort_log"] = stats.pearsonr(df["log_gdp"], df["log_child_mort"])[0]
by_decade = pd.Series({dec: stats.pearsonr(g["log_gdp"], g["life_exp"])[0] for dec, g in df.groupby("decade")})
r["decade_low"], r["decade_high"] = by_decade.min(), by_decade.max()

In [7]:
# test 1: ANOVA
order = ["Q1 poorest", "Q2", "Q3", "Q4 richest"]
groups = [df[df["gdp_quartile"] == q]["life_exp"] for q in order]
r["f_anova"], r["p_anova"] = stats.f_oneway(*groups)
grand = df["life_exp"].mean()
r["eta"] = sum(len(g) * (g.mean() - grand) ** 2 for g in groups) / ((df["life_exp"] - grand) ** 2).sum()
r["q_means"] = [g.mean() for g in groups]


In [8]:
# test 2: chi-square
table = pd.crosstab(df["gdp_group"], df["mort_group"])
r["chi2"], r["p_chi"], r["dof"], _ = stats.chi2_contingency(table)
r["v"] = np.sqrt(r["chi2"] / (table.values.sum() * (min(table.shape) - 1)))
r["low_mort_rich"] = table.loc["Above median GDP", "Low child mortality"] / table.loc["Above median GDP"].sum() * 100
r["low_mort_poor"] = table.loc["Below median GDP", "Low child mortality"] / table.loc["Below median GDP"].sum() * 100


In [9]:
# test 3: before and after 2000
before = df[df["period"] == "Before 2000"]["life_exp"]
after = df[df["period"] == "2000 and after"]["life_exp"]
r["before_mean"], r["after_mean"] = before.mean(), after.mean()
r["t_ab"], r["p_ab"] = stats.ttest_ind(before, after, equal_var=False)
r["diff"] = r["after_mean"] - r["before_mean"]
pooled = np.sqrt(((len(before) - 1) * before.var() + (len(after) - 1) * after.var()) / (len(before) + len(after) - 2))
r["d"] = r["diff"] / pooled
cm = df.groupby(["Entity", "period"])["life_exp"].mean().unstack().dropna()
r["countries_up"], r["countries_both"] = int((cm["2000 and after"] > cm["Before 2000"]).sum()), len(cm)


In [10]:
# probability model
df["long_life"] = (df["life_exp"] >= 70).astype(int)
model = sm.Logit(df["long_life"], sm.add_constant(df["log_gdp"])).fit(disp=False)
r["odds_double"] = np.exp(model.params["log_gdp"] * np.log(2))
r["half_dollars"] = np.exp(-model.params["const"] / model.params["log_gdp"])
r["accuracy"] = ((model.predict(sm.add_constant(df["log_gdp"])) >= 0.5).astype(int) == df["long_life"]).mean()
r["baseline"] = max(df["long_life"].mean(), 1 - df["long_life"].mean())


In [11]:
# the spread at the same income (log GDP between 8.5 and 9)
same = df[(df["log_gdp"] > 8.5) & (df["log_gdp"] < 9)]["life_exp"]
r["same_low"], r["same_high"] = same.quantile(0.05), same.quantile(0.95)


In [12]:
# the flattening in 2019
y19 = df[df["Year"] == 2019]
low_band = y19[y19["gdp_pc"] < 2000]["life_exp"].mean()
high_band = y19[y19["gdp_pc"] > 40000]["life_exp"].mean()
r["low_band"], r["high_band"] = low_band, high_band


In [13]:
# Kenya in 2019
d = y19.copy()
slope, intercept = np.polyfit(d["log_gdp"], d["life_exp"], 1)
k = d[d["Entity"] == "Kenya"].iloc[0]
r["k_gdp"], r["k_life"], r["k_mort"] = k["gdp_pc"], k["life_exp"], k["child_mort"]
r["k_gap"] = k["life_exp"] - (slope * k["log_gdp"] + intercept)
r["typical_life"] = y19["life_exp"].median()

print("Numbers ready:", len(r))

Numbers ready: 40


# Findings Summary Wealth And Health, 1950 to 2022

## At a Glance

### Question

Does a higher GDP per capita go with longer life expectancy and lower child mortality, and has that link held since 1950?

### Short Answer

Yes, strongly, and in every decade. But income does not explain everything, and the gains flatten at high income.

### Hypothesis

Richer countries are healthier, but the gains flatten and income alone does not explain everything. The evidence is consistent with all three parts.

### Decision it Supports

Growth alone is not a safe plan. The data suggests also funding health directly, especially at lower incomes.

### Local Case

Kenya: life expectancy rose 14.5 years from 1960 to 2019, but in 2019 it was 2.7 years below what its income predicts.

---

## The Data

Raw rows show life expectancy at 21,565, child mortality at 17,066, and GDP per capita at 21,586. After removing regions and groups, life expectancy is 19,452, child mortality is 16,161, and GDP per capita is 20,934.

Joining life expectancy and GDP gives 13,218 rows across 165 countries. Adding child mortality yields 11,828 rows across 163 countries. After keeping 1950 onward, the final table contains 10,429 rows across 163 countries.

The final table has one row per country per year, 1950 to 2022, with no missing values and no duplicates. Coverage before 1950 is thin and uneven.

---

## What I Found

### The Shape of the Data

Life expectancy is skewed left with a skew of -0.70 where most country-years are high, with a tail of low values, fitting approximately normal with 67.6%, 95.8%, and 99.8% within 1, 2, and 3 standard deviations.

Child mortality is skewed right with a skew of 1.54 where most are low with a tail of high values, and does not fit a normal curve.

GDP per capita is strongly skewed right with a skew of 3.16 and does not fit a normal curve.

Log GDP per capita is roughly symmetric with a skew of 0.12 and fits approximately normal with a flat top. Log GDP is used in the analysis because it fixes the skew.

### Correlations and Modelling

Log GDP and life expectancy have a Pearson r of 0.83 and Spearman of 0.86, showing a strong link where income explains about 69% of the differences.

Log GDP and child mortality have a Pearson r of -0.74 and Spearman of -0.86, showing that richer country-years lose fewer children.

Log GDP and log child mortality have a Pearson r of -0.85 and Spearman of -0.86, which is the strongest because both are on a log scale.

Raw GDP and life expectancy have a Pearson r of 0.62 and Spearman of 0.86, which is weaker because the relationship is curved.

The correlation has held over time, with correlation with life expectancy ranging from 0.78 to 0.88 and with child mortality ranging from -0.71 to -0.84 in every decade since 1950. In single years with one dot per country, 1960 was 0.74, 1990 was 0.89, and 2019 was 0.84.

### Probability and Modelling

The chance of low child mortality given high GDP is 0.855.

The chance of low child mortality given low GDP is 0.146, making high GDP 5.9 times as likely.

An independence check expected 0.25 if unrelated, but actual is 0.427, showing they are not independent.

The chance of a life expectancy of 70+ by GDP quartile from poorest to richest is 0.6%, 10.4%, 52.7%, and 90.3%.

In the logistic model for doubling income, odds of 70+ years are 5.7 times higher.

The 50% point in the logistic model is about 8,800 dollars per person.

The logistic model accuracy is 85.8% against 61.5% for always guessing the most common outcome.

### The Three Hypothesis Tests and Evaluation

Test 1 ANOVA tests if average life expectancy is the same in all four GDP groups, yielding F = 6,339.31 and p < 0.001 with averages of 50.1, 60.4, 69.0, and 75.6 years, rejecting H0 with an effect size Eta squared = 0.646 which is large.

Test 2 Chi-square tests if GDP group and child mortality group are independent, yielding Chi-square = 5,246.5 with 1 degree of freedom and p < 0.001 where 85.5% versus 14.6% have low child mortality, rejecting H0 with Cramer's V = 0.71 which is strong.

Test 3 A/B-style Welch t-test tests if average life expectancy is the same before and after 2000, yielding t = -47.59 and p < 0.001 with 60.3 before and 70.1 after, rejecting H0 with Cohen's d = 0.90 which is large.

Extra checks include Tukey pairs showing every pair of groups differs as gaps shrink to 10.3, 8.6, and then 6.5 years. Kruskal-Wallis yields H = 7,198 and p < 0.001. Country averages yield F = 267.5 and p < 0.001. Test 3 difference with 95% interval is 9.84 years from 9.44 to 10.25. Country by country analysis shows all 163 countries have both periods with an average gain of 9.60 years, and 158 of 163 improved with paired t = 24.11 and p < 0.001.

A p-value is not the probability that H0 is true, and it does not show how big a difference is, which is why effect sizes are reported.

Test 3 caution notes that the two periods were not randomly assigned, so it shows that life expectancy differs, not what caused the change.

### Kenya Local Case

In 1960 Kenya income per person was 1,157 international dollars, life expectancy was 48.5 years, and child mortality data is unavailable.

In 1990 Kenya income per person was 1,780 international dollars, life expectancy was 58.3 years.

In 2019 Kenya income per person was 3,207 international dollars, life expectancy was 62.9 years, and child mortality was 4.27 compared to a typical country 2019 income of 12,350, life expectancy of 73.8, and child mortality of 1.56.

Kenya compared with what its income predicts shows it was above prediction for 38 of 73 years, with a biggest lead of +7.9 years in 1984, a biggest shortfall of -5.8 years in 1951, a 2019 gap of -2.7 years, and a rank among the 9 countries with the most similar income in 2019 of 6th for life expectancy where Syria was 71.0 top and Cote d'Ivoire was 60.3 bottom.

Kenya ran ahead of its income-predicted line from the late 1950s to about the early 1990s, then fell below it, with data showing when, not why.

---

## Was My Hypothesis Right

### Richer Countries

Longer life expectancy and lower child mortality with r = 0.83 and -0.74 to -0.85, and all three tests reject H0, which is consistent with the data.

### Link Over Time

The link has held since 1950 with correlation 0.78 to 0.88 in every decade, which is consistent with the data.

### Flattening Gains

The gains flatten at high income with raw GDP curve bends at Pearson 0.62 versus Spearman 0.86, 2019 bands gaining +5.4, +4.1, +4.6, +4.0, then +3.3 years, and quartile gaps shrinking to 10.3, 8.6, 6.5, which is consistent with the data.

### Income Independence

Income alone does not explain everything, as income explains about 69% of the differences and at the same income from log GDP 8.5 to 9 life expectancy runs from 54 to 74 years, a spread of about 20 years, which is consistent with the data.

---

## What This Means for the Decision

Wealth and health move together strongly and steadily, but at the same income countries differ by about 20 years, and each extra step up in income adds fewer years.

A government or donor should not rely on growth alone and should also invest directly in health, especially at lower incomes where the gap between countries is widest.

This is a judgement from patterns, not a proof, and the data cannot show that spending on health would close the gap.

---

## Limits

### Correlation and Causation

Correlation is not causation because wealth may improve health, health may improve wealth, or something else such as education or peace may drive both.

### Independence of Rows

The same country appears about 64 times so rows are not independent, making p-values look stronger than they should, which was addressed using effect sizes and country-level checks.

### Grouping Choices

Groups are choices since median splits, the 70-year cut-off, and before or after 2000 were not randomly assigned.

### Median Pooling

Median splits pool all years so high GDP partly means a later year.

### Inequality

Averages hide inequality because a national figure says nothing about differences inside a country.

### Health Spending Missing

No health spending is in the data so it can show how much income leaves unexplained, not what fills the gap.

### Child Mortality Unit

Child mortality unit is not yet confirmed so axis labels need the right unit from the source page.

---

## Short Version for the README, Slides and Article

Across 163 countries from 1950 to 2022, richer countries have lived longer and lost fewer young children in every decade with a correlation of 0.83 with life expectancy. Three hypothesis tests all rejected the idea of no difference, showing the richest quarter of country-years averaged about 76 years against 50 for the poorest, and average life expectancy rose almost 10 years after 2000. But the gains flatten at high income, and countries with the same income differ by about 20 years, so growth alone is not enough. In Kenya, life expectancy rose 14.5 years from 1960 to 2019, yet in 2019 it was still about 2.7 years below what its income would predict. This shows patterns, not causes.